In [7]:
%reset -f
import gdsfactory as gf
import import_ipynb
from datetime import datetime
import sys
sys.path.append("/Users/bubble/Desktop/Project/Buckling/non_hookean/Layout")

from gdsfactory.generic_tech import get_generic_pdk
PDK = get_generic_pdk()
PDK.activate()
# %run "/Users/bubble/Desktop/Project/T_sensor/T_sensor/cell/cell_mesh.ipynb"


In [8]:
'''
Layer description:
1: Solid beam / thin down (optical lithography)
2: Gold deposition
3: KOH etching (before mirror)
4: KOH etching (after mirror)
5: Hinge / solid beam (Ebeam lithography)
6: Marker layer for alignment
7: Hole etching (optical lithography)
10,20,21: Wafer edge and piece edges 
'''

'\nLayer description:\n1: Solid beam / thin down (optical lithography)\n2: Gold deposition\n3: KOH etching (before mirror)\n4: KOH etching (after mirror)\n5: Hinge / solid beam (Ebeam lithography)\n6: Marker layer for alignment\n7: Hole etching (optical lithography)\n10,20,21: Wafer edge and piece edges \n'

In [9]:
def die_marker(component, x, y, die_type, order):
    for i in range(4):
        marker = gf.components.text(f"{die_type}_{order+i}", size=1000, layer=(3, 0))
        (component << marker).movex(x+2500).movey(y+(i-2)*4750 + 2000)
    



In [10]:
from cell_width import cell_width
from cell_length import cell_length
from cell_size import cell_size
from cell_thickness_test_squre import cell_thickness_test_square
from cell_thickness_test_groove import cell_thickness_test_groove

wafer = gf.Component()


# diff width
wafer.add_ref(cell_width).movex(0).movey(20000)
die_marker(wafer, 0, 20000, 'width', 1)
wafer.add_ref(cell_width).movex(0).movey(0)
die_marker(wafer, 0, 0, 'width', 5)
wafer.add_ref(cell_width).movex(0).movey(-20000)
die_marker(wafer, 0, -20000, 'width', 9)
wafer.add_ref(cell_width).movex(20000).movey(20000)
die_marker(wafer, 20000, 20000, 'width', 13)
wafer.add_ref(cell_width).movex(20000).movey(0)
die_marker(wafer, 20000, 0, 'width', 17)
wafer.add_ref(cell_width).movex(20000).movey(-20000)
die_marker(wafer, 20000, -20000, 'width', 21)

# diff length
wafer.add_ref(cell_length).movex(-20000).movey(20000)
die_marker(wafer, -20000, 20000, 'length', 1)
wafer.add_ref(cell_length).movex(-20000).movey(0)
die_marker(wafer, -20000, 0, 'length', 5)
wafer.add_ref(cell_length).movex(-20000).movey(-20000)
die_marker(wafer, -20000, -20000, 'length', 9)

# diff size
wafer.add_ref(cell_size).movex(-40000).movey(20000)
die_marker(wafer, -40000, 20000, 'size', 1)
wafer.add_ref(cell_size).movex(-40000).movey(0)
die_marker(wafer, -40000, 0, 'size', 5)
wafer.add_ref(cell_size).movex(-40000).movey(-20000)
die_marker(wafer, -40000, -20000, 'size', 9)


# etching thickness test
wafer.add_ref(cell_thickness_test_square).movey(40000)
wafer.add_ref(cell_thickness_test_groove).movey(-35830)
wafer.add_ref(cell_thickness_test_groove).movex(-20000).movey(40000)
wafer.add_ref(cell_thickness_test_square).movex(-20000).movey(-35830)

# wafer.show()

Unnamed_260: ports [], KCell(name=Unnamed_201, ports=[], pins=[], instances=['Unnamed_196_2500000_-7500000'], locked=False, kcl=DEFAULT)

In [11]:
# grid
width = 750/2
circle = gf.components.circle(50000, layer=(10, 0))
circle_ref = wafer << circle
# horizontal lines of the grid
h_lines = [2, 4, 4, 4, 4, 4, 4, 4, 2]
x_size = [1, 1, 1, 1, 1, 1, 1, 1, 1]
y_space = [0, 15830, 10000, 10000, 10000, 10000, 10000, 10000, 15830]
x_temp = [ -2, -1, 0, 1]
x_space = [[-1, 0], x_temp, x_temp, x_temp, x_temp, x_temp, x_temp, x_temp, [-1, 0]]
gap = 600 # um
y_space_temp = 0
for i in range(len(h_lines)):
    y_space_temp += y_space[i]
    for j in range(h_lines[i]):
        horizontal_line = gf.components.rectangle(size=(20000*x_size[i]-2*gap, width), layer=(3, 0))
        frame = wafer << horizontal_line
        frame.move((x_space[i][j]*20000+gap, 45830-y_space_temp-width/2))
# vertical lines of the grid
style1 = [10000] * 6
style2 = [15830] + [10000]*6 + [15830]
v_lines_len = [style1, style2, style2, style2, style1]
y_top = [20000, 30000, 30000, 30000, 20000]

for i in range(len(v_lines_len)):
    y_space_temp = y_top[i]
    for j in range(len(v_lines_len[i])):
        x_pos = (i-2)*20000 - width/2
        vertical_line = gf.components.rectangle(size=(width, v_lines_len[i][j]-2*gap), layer=(3, 0))
        frame = wafer << vertical_line
        frame.move((x_pos,y_space_temp+gap))
        if j != len(v_lines_len[i])-1:
            y_space_temp -= v_lines_len[i][j+1]


# side grids





In [12]:
mark_temp = gf.Component()
# marker
marker_1 = gf.components.rectangle(size=(1, 1000), layer=(1, 0))
marker_2 = gf.components.rectangle(size=(1000, 1), layer=(1, 0))
marker_basic = gf.Component()
marker_1_ref = marker_basic << marker_1
marker_2_ref = marker_basic << marker_2
marker_1_ref.move((-0.5, -500))
marker_2_ref.move((-500, -0.5))
marker_tri = gf.Component()
marker_tri.add_ref(marker_basic).movex(-1000)
marker_tri.add_ref(marker_basic).movey(1000)
marker_tri.add_ref(marker_basic).movey(-1000)
mark_left = mark_temp << marker_tri
mark_left.move((-40700, 0))
mark_right = mark_temp << marker_tri
mark_right.dmirror_x(0).movex(40700)
# mark position
''''
mark1: (-41700, 0), (-40700, 1000), (-40700, -1000)
mark2: (41700, 0), (40700, 1000), (40700, -1000)
'''''
# left label
label_left1 = gf.components.text("(-41700, 0)", size=50, layer=(1, 0))
label_left2 = gf.components.text("(-40700, 1000)", size=50, layer=(1, 0))
label_left3 = gf.components.text("(-40700, -1000)", size=50, layer=(1, 0))
label_left = gf.Component()
label_left.add_ref(label_left1).move((-200, 600))
label_left.add_ref(label_left2).move((700, 1600))
label_left.add_ref(label_left3).move((700, -400))
label_left_ref = mark_temp << label_left
label_left_ref.move((-41700, 0))
# right label
label_right1 = gf.components.text("(41700, 0)", size=50, layer=(1, 0))
label_right2 = gf.components.text("(40700, 1000)", size=50, layer=(1, 0))
label_right3 = gf.components.text("(40700, -1000)", size=50, layer=(1, 0))
label_right = gf.Component()
label_right.add_ref(label_right1).move((-200, 600))
label_right.add_ref(label_right2).move((-1200, 1600))
label_right.add_ref(label_right3).move((-1200, -400))
label_right_ref = mark_temp << label_right
label_right_ref.move((41700, 0))

wafer << mark_temp


# ATTENTION
# mirror the backside pattern for backside photolithography
wafer_backside = gf.boolean(A = wafer, B = wafer,  operation="not", layer1=(3, 0), layer2=(50, 0), layer=(4, 0))
wafer.add_ref(wafer_backside).dmirror_x(0)
wafer.show()
suffix = datetime.now().strftime("%Y%m%d_%H%M")
# wafer.write_gds(f"wafer_ID82{suffix}.gds", with_metadata=False)
